# Pharmacy Invoice Reconciliation

Автоматизация сверки документов между файлами СБИС и аптечными выгрузками.

Логика файла:
1) собираем и объединяем все CSV из СБИС;
2) для каждой аптечной выгрузки ищем соответствующий документ;
3) заполняем номер и дату счет-фактуры, сумму и флаг совпадения дат;
4) сохраняем итог в Excel.

## Описание задачи:

* Ваша компания «переезжает» с упрощенной системы налогообложения на общую. К вам подошел главный бухгалтер с просьбой:

"В нашей сети около 100 аптек. Нам нужно единоразово загрузить файл в 1С, но для этого надо сделать сравнение реализации по аптекам и выгрузки из СБИСа. Пункты, которые расходятся, мы подправим вручную. После этого будем загружать в 1С. Подробная информация ниже."

* Выгрузка из СБИС находится в папке Входящие. Загрузите все csv файлы в один датафрейм. Если встретился файл с другим расширением, то его пропускаем. Будьте аккуратны с заголовками столбцов - не мешайте их с данными при объединении нескольких файлов. 

* Дайте столбцам полученного датафрейма такие названия:

In [1]:
"Дата",
"Номер",
"Сумма",
"Статус",
"Примечание",
"Комментарий",
"Контрагент",
"ИНН/КПП",
"Организация",
"ИНН/КПП",
"Тип документа",
"Имя файла",
"Дата",
"Номер 1",
"Сумма 1",
"Сумма НДС",
"Ответственный",
"Подразделение",
"Код",
"Дата",
"Время",
"Тип пакета",
"Идентификатор пакета",
"Запущено в обработку",
"Получено контрагентом",
"Завершено",
"Увеличение суммы",
"НДC",
"Уменьшение суммы",
"НДС"

'НДС'

* Все названия из нескольких слов сцепите через нижнее подчеркивание в одно_длинное_слово.

* Файлы с выгрузкой из аптек находятся в папке Аптеки/csv/correct/.

* Загружайте на обработку по одному файлу. Содержимое каждого файла загружайте в датафрейм Pandas. Если встречаете не csv формат - игнорируйте его.

* Создайте столбцы:

In [2]:
"Номер счет-фактуры"
"Сумма счет-фактуры"
"Дата счет-фактуры"
"Сравнение дат"

'Сравнение дат'

* В каждой строке проверьте:

1. Если "Поставщик" - ЕАПТЕКА, то к "Номер накладной" нужно добавить /15.
2. Нужно найти все записи в выгрузке из СБИСа по данному номеру накладной
3. Из найденных строк нужно оставить только те, которые имеют один из типов документа: ["СчФктр", "УпдДоп", "УпдСчфДоп", "ЭДОНакл"]
4. Если ничего не найдено - просто переходим к следующей строке
5. Если найдено - нужно сохранить значения Номер, Сумма и Дата
6. Дату нужно представить в формате 25.05.2021
7. В столбцы из пункта 5 нужно записать найденные для данной строки значения
8. В столбец Сравнение дат помещаем "Не совпадает!", если найденная дата и дата накладной отличаются. Иначе - пустая строка.

Обратите внимание: если по заданным условиям находится несколько подходящих документов - выбираем значения из первого найденного. Это означает, что если у документов разные суммы, берется та, что стоит первой в выгрузке из СБИС.

Если дата счет фактуры отсутствует или пуста, а дата накладной заполнена, такая ситуация считается расхождением.

* Итоговый файл выгрузки по аптеке должен содержать такие столбцы в заданном порядке:

In [3]:
'№ п/п', 'Штрих-код партии', 'Наименование товара', 'Поставщик',
'Дата приходного документа', 'Номер приходного документа',
'Дата накладной', 'Номер накладной', 'Номер счет-фактуры',
'Сумма счет-фактуры', 'Кол-во',
'Сумма в закупочных ценах без НДС', 'Ставка НДС поставщика',
'Сумма НДС', 'Сумма в закупочных ценах с НДС', 'Дата счет-фактуры', 'Сравнение дат'

('Сумма НДС',
 'Сумма в закупочных ценах с НДС',
 'Дата счет-фактуры',
 'Сравнение дат')

* Файл сохраните по пути 

Результат/{полная сегодняшняя дата}/{имя исходного файла без расширения} - результат.xlsx. 

Если таких папок не существует - создайте их (проверку и создание нужно реализовать средствами Python, автоматически).

## Решение:

In [4]:
from datetime import datetime

In [5]:
import pandas as pd
from pathlib import Path

## 1. Пути и выходная папка

Результаты сохраняются в подпапку output/текущая_дата.

In [6]:
output_dir = Path("output") / datetime.now().strftime("%Y-%m-%d")
output_dir.mkdir(parents=True, exist_ok=True)

## 2. Список колонок для объединенной выгрузки СБИС

После чтения CSV без заголовков вручную присваиваем имена столбцам.

In [7]:
sbis_columns = [
    "Дата",
    "Номер",
    "Сумма",
    "Статус",
    "Примечание",
    "Комментарий",
    "Контрагент",
    "ИНН/КПП",
    "Организация",
    "ИНН/КПП",
    "Тип документа",
    "Имя файла",
    "Дата",
    "Номер 1",
    "Сумма 1",
    "Сумма НДС",
    "Ответственный",
    "Подразделение",
    "Код",
    "Дата",
    "Время",
    "Тип пакета",
    "Идентификатор пакета",
    "Запущено в обработку",
    "Получено контрагентом",
    "Завершено",
    "Увеличение суммы",
    "НДC",
    "Уменьшение суммы",
    "НДС",
]

## 3. Чтение и объединение файлов СБИС

In [8]:
sbis_dir = Path('incoming_files')
sbis_file_names = [file.name for file in sbis_dir.iterdir()]

In [9]:
sbis_parts = []

Берем только CSV-файлы.

skiprows=1: пропускаем первую строку,

sep=';' и encoding='windows-1251' соответствуют формату выгрузки.

In [10]:
for file_name in sbis_file_names:
    if 'csv' not in file_name:
        continue
    sbis_part_df = pd.read_csv(
        sbis_dir / file_name,
        skiprows=1,
        sep=';',
        encoding='windows-1251',
        header=None,
    )
    sbis_parts.append(sbis_part_df)

Склеиваем все куски в одну таблицу.

In [11]:
sbis_df = pd.concat(sbis_parts, ignore_index=True)

Проставляем заголовки и сразу заменяем пробелы на "_",
чтобы к столбцам было удобнее обращаться в коде.

In [12]:
sbis_df.columns = sbis_columns
sbis_df.columns = [column.replace(' ', '_') for column in sbis_df.columns]

## 4. Чтение аптечных файлов

In [13]:
pharmacy_dir = Path('pharmacy_files') / 'csv' / 'correct'
pharmacy_file_names = [file.name for file in pharmacy_dir.iterdir()]

Типы документов, которые считаем подходящими при поиске в СБИС.

In [14]:
valid_document_types = ["СчФктр", "УпдДоп", "УпдСчфДоп", "ЭДОНакл"]

## 5. Обработка каждой аптечной выгрузки

In [ ]:
for file_name in pharmacy_file_names:
    if 'csv' not in file_name:
        continue
    pharmacy_df = pd.read_csv(pharmacy_dir / file_name, sep=';', encoding='windows-1251')
    
    #Добавляем новые колонки, которые будем заполнять по результатам сверки.
    pharmacy_df["Номер счет-фактуры"] = ""
    pharmacy_df["Сумма счет-фактуры"] = ""
    pharmacy_df["Дата счет-фактуры"] = ""
    pharmacy_df["Сравнение дат"] = ""

    #Проходим по строкам аптечной выгрузки и ищем соответствующий документ в СБИС.
    for row_index, row in pharmacy_df.iterrows():
        invoice_number = row["Номер накладной"]

        #Специальное бизнес-правило: для поставщика ЕАПТЕКА к номеру накладной добавляется суффикс /15.
        if 'ЕАПТЕКА' in row["Поставщик"]:
            invoice_number += "/15"

        #Ищем записи в СБИС с нужным номером.
        matched_records_df = sbis_df[sbis_df.Номер.values == invoice_number]
        matched_records_df = matched_records_df[matched_records_df.Тип_документа.isin(valid_document_types)]

        #Оставляем только документы нужных типов.
        if matched_records_df.empty:
            continue
            
        #Берем первую найденную запись.
        matched_invoice_number = matched_records_df.iloc[0]["Номер"]
        matched_invoice_sum = matched_records_df.iloc[0]["Сумма"]
        
        #В исходной выгрузке дата хранится как строка. Здесь сохраняем исходную логику преобразования.
        matched_invoice_date = matched_records_df.iloc[0]["Дата"][1]
        matched_invoice_date = datetime.strptime(matched_invoice_date, "%d.%m.%y").strftime("%d.%m.%Y")

        #Заполняем результат сверки в аптечной таблице.
        pharmacy_df.at[row_index, "Номер счет-фактуры"] = matched_invoice_number
        pharmacy_df.at[row_index, "Сумма счет-фактуры"] = matched_invoice_sum
        pharmacy_df.at[row_index, "Дата счет-фактуры"] = matched_invoice_date

        #Если дата счет-фактуры и дата накладной не совпали, ставим пометку для дальнейшей ручной проверки.
        pharmacy_df.at[row_index, "Сравнение дат"] = (
            ""
            if matched_invoice_date == pharmacy_df.at[row_index, 'Дата накладной']
            else "Не совпадает!"
        )

        #Оставляем в выходном файле только нужные колонки и в удобном порядке.
        result_columns = [
        '№ п/п',
        'Штрих-код партии',
        'Наименование товара',
        'Поставщик',
        'Дата приходного документа',
        'Номер приходного документа',
        'Дата накладной',
        'Номер накладной',
        'Номер счет-фактуры',
        'Сумма счет-фактуры',
        'Кол-во',
        'Сумма в закупочных ценах без НДС',
        'Ставка НДС поставщика',
        'Сумма НДС',
        'Сумма в закупочных ценах с НДС',
        'Дата счет-фактуры',
        'Сравнение дат',
        ]
        pharmacy_df = pharmacy_df[result_columns]
        
        #Сохраняем Excel-файл с понятным названием.
        output_file_name = f"{file_name.split('.csv')[0]} - результат.xlsx"
        pharmacy_df.to_excel(output_dir / output_file_name, index=False)
    
    print(f'{file_name} обработан')

C:\Users\Zver\AppData\Local\Temp\ipykernel_3488\126100506.py:33: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  matched_invoice_date = matched_records_df.iloc[0]["Дата"][1]


366.csv обработан


C:\Users\Zver\AppData\Local\Temp\ipykernel_3488\126100506.py:33: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  matched_invoice_date = matched_records_df.iloc[0]["Дата"][1]


366.csv обработан


C:\Users\Zver\AppData\Local\Temp\ipykernel_3488\126100506.py:33: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  matched_invoice_date = matched_records_df.iloc[0]["Дата"][1]


...

C:\Users\Zver\AppData\Local\Temp\ipykernel_3488\126100506.py:33: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  matched_invoice_date = matched_records_df.iloc[0]["Дата"][1]


366.csv обработан


C:\Users\Zver\AppData\Local\Temp\ipykernel_3488\126100506.py:33: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  matched_invoice_date = matched_records_df.iloc[0]["Дата"][1]


366.csv обработан


C:\Users\Zver\AppData\Local\Temp\ipykernel_3488\126100506.py:33: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  matched_invoice_date = matched_records_df.iloc[0]["Дата"][1]


---
**Примечание:** для запуска ноутбука достаточно открыть его в корне соответствующего репозитория — все файлы данных лежат рядом и читаются по относительному пути.